# TP1/TP2 Longitudinal Analysis - Graphs

This file contains the scatterplot and model fit for the TP1/TP1 longitudinal analysis of canonical proportion (CP) as a predictor of the following speech language measures:
- PPVT
- CTOPP
- NWR
- RWR

The weighted, scaled models are used on the scaled outcome measures.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import to_rgba
from sklearn.preprocessing import StandardScaler

# === Load TP1/TP2 file with scaled columns ===
df = pd.read_csv("tp1_tp2_master_with_scaled.csv")

# --- Weights: num_clips = 0 + 1 (mean-normalized, lower bound 1) ---
df["num_clips"] = pd.to_numeric(df.get("0", 0), errors="coerce").fillna(0) + \
                  pd.to_numeric(df.get("1", 0), errors="coerce").fillna(0)
w = df["num_clips"].astype(float).clip(lower=1)
w = w / w.mean()

# --- Robust gender dummy: 1=Female, 0=Male ---
def to_gender_dummy(x):
    try:
        v = float(x)
        if v in (0.0, 1.0):
            return int(v)
    except Exception:
        pass
    s = str(x).strip().lower()
    if s in ("f", "female"): return 1
    if s in ("m", "male"):   return 0
    return np.nan

df["gender_dummy"] = df.get("gender", np.nan).apply(to_gender_dummy)

# --- Coerce & scale covariates ---
for col in ["age_mos", "Maternal_Education_Level", "canonical_proportion_scaled"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

sc_age = StandardScaler()
sc_med = StandardScaler()
df["age_mos_scaled"] = sc_age.fit_transform(df[["age_mos"]])
df["Maternal_Education_Level_scaled"] = sc_med.fit_transform(df[["Maternal_Education_Level"]])

# --- Helper: WLS with controls; predictions vary CP, hold controls at means ---
def fit_weighted_with_controls(y_col, data, weights_series):
    predictors = [
        "canonical_proportion_scaled",
        "age_mos_scaled",
        "gender_dummy",
        "Maternal_Education_Level_scaled",
    ]
    needed = [y_col, "num_clips"] + predictors
    keep = data[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if keep.empty or keep.shape[0] < 3:
        return None

    # Always add a constant during fit
    X = sm.add_constant(keep[predictors], has_constant="add")
    y = keep[y_col]
    ww = weights_series.loc[keep.index].to_numpy()
    model = sm.WLS(y, X, weights=ww).fit()

    # Controls at means (gender = proportion female)
    ctrl_means = {
        "age_mos_scaled": float(keep["age_mos_scaled"].mean()),
        "gender_dummy": float(keep["gender_dummy"].mean()),
        "Maternal_Education_Level_scaled": float(keep["Maternal_Education_Level_scaled"].mean()),
    }

    return model, keep, ctrl_means

# --- Outcomes to plot: (column, panel tag, y-axis label, color) ---
outcomes = [
    ("PPVT_Standard_scaled",        "A", "PPVT-4 (SS)",  "#ffd500"),  # yellow
    ("CTOPP_Elision_Scaled_scaled", "B", "CTOPP-2 (SS)", "#2ca02c"),  # green
    ("avg_rwr_accuracy_scaled",     "C", "RWR Accuracy", "#d62728"),  # red
    ("avg_nwr_accuracy_scaled",     "D", "NWR Accuracy", "#9467bd"),  # purple
]

# --- Fit all models ---
fits = []
for col, *_ in outcomes:
    if col not in df.columns:
        print(f"[skip] Missing column: {col}")
        fits.append(None)
    else:
        fits.append(fit_weighted_with_controls(col, df, w))

# --- Global CP grid & x-limits (all panels share) ---
cp_all = df["canonical_proportion_scaled"].dropna()
x_lo, x_hi = float(cp_all.min()), float(cp_all.max())
xlim = (x_lo, x_hi)
cp_grid = np.linspace(xlim[0], xlim[1], 400)

# Build prediction matrix that EXACTLY matches model exog (fixes shape mismatch)
def make_pred_X(model, ctrl_means, cp_vals):
    # Model’s expected column order (including 'const')
    exog_names = list(model.model.exog_names)

    # Build dict for all non-const predictors
    base = {
        "canonical_proportion_scaled": cp_vals,
        "age_mos_scaled": ctrl_means["age_mos_scaled"],
        "gender_dummy": ctrl_means["gender_dummy"],
        "Maternal_Education_Level_scaled": ctrl_means["Maternal_Education_Level_scaled"],
    }

    # Create DF with exactly the model's columns (excluding 'const' for now)
    non_const = [c for c in exog_names if c != "const"]
    Xp = pd.DataFrame({c: base[c] for c in non_const})
    # Force add const, then reorder to EXACT exog_names
    Xp = sm.add_constant(Xp, has_constant="add")
    Xp = Xp[exog_names]
    return Xp

# --- Figure (same layout/style) ---
plt.rcParams.update({
    "axes.labelsize": 16,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

fig = plt.figure(figsize=(13.2, 7.6), constrained_layout=False)
gs = GridSpec(nrows=2, ncols=3, figure=fig,
              width_ratios=[1, 1, 0.42], height_ratios=[1, 1],
              wspace=0.01, hspace=0.25)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
]
ax_leg = fig.add_subplot(gs[:, 2]); ax_leg.axis("off")

ylabel_to_col = {
    "PPVT-4 (SS)":        "PPVT_Standard_scaled",
    "CTOPP-2 (SS)":       "CTOPP_Elision_Scaled_scaled",
    "RWR Accuracy":       "avg_rwr_accuracy_scaled",
    "NWR Accuracy":       "avg_nwr_accuracy_scaled",
}

def plot_panel(ax, fit_tuple, color, panel_tag, ylabel):
    if fit_tuple is None:
        ax.set_visible(False); return
    model, keep, ctrl_means = fit_tuple
    y_col = ylabel_to_col[ylabel]

    # sizes & opacity by clip counts
    sizes = np.clip(keep["num_clips"]/40.0, 20, 600)
    cmin, cmax = keep["num_clips"].min(), keep["num_clips"].max()
    alphas = 0.35 + 0.65 * (keep["num_clips"] - cmin) / (cmax - cmin + 1e-9)
    r, g, b, _ = to_rgba(color)
    facecols = [(r, g, b, a) for a in alphas]

    # Scatter
    ax.scatter(keep["canonical_proportion_scaled"], keep[y_col],
               s=sizes, edgecolor="k", linewidth=0.35, c=facecols)

    # Predictions with controls held at means (MATCH model exog)
    Xp = make_pred_X(model, ctrl_means, cp_grid)
    pred = model.get_prediction(Xp).summary_frame(alpha=0.05)
    ax.plot(cp_grid, pred["mean"], color=color, linewidth=2.6)
    ax.fill_between(cp_grid, pred["mean_ci_lower"], pred["mean_ci_upper"],
                    color=color, alpha=0.25)

    # Labels & annotations
    ax.set_xlabel("Canonical Proportion (Scaled)")
    ax.set_ylabel(ylabel)
    ax.text(0.03, 0.92, f"({panel_tag})", transform=ax.transAxes,
            fontsize=16, fontweight="bold")
    if "canonical_proportion_scaled" in model.params.index:
        ax.text(0.05, 0.06, f"β = {model.params['canonical_proportion_scaled']:.2f}",
                transform=ax.transAxes, fontsize=14, fontweight="bold")

    # Limits & square shape
    ax.set_xlim(*xlim); ax.margins(x=0)
    q1, q99 = keep[y_col].quantile([0.01, 0.99])
    y_pad = 0.10 * (q99 - q1 + 1e-9)
    ax.set_ylim(float(q1 - y_pad), float(q99 + y_pad))
    ax.set_box_aspect(1)

    # Borders
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(1)

# --- Draw panels ---
for ax, fit_tuple, (col, tag, ylab, color) in zip(axes, fits, outcomes):
    plot_panel(ax, fit_tuple, color, tag, ylab)

# --- Legend (clip sizes) ---
clip_sizes = [50, 200, 1000, 3000, 5000, 8000]
handles = [ax_leg.scatter([], [], s=np.clip(s/40.0, 20, 600), color="gray",
                          alpha=0.7, edgecolor="k", linewidth=0.35)
           for s in clip_sizes]
labels = [f"= {s} clips" for s in clip_sizes]
ax_leg.legend(handles, labels, loc="center left", frameon=True,
              borderpad=1.6, labelspacing=1.6, handletextpad=1.2,
              borderaxespad=0.6, fontsize=12, title="Clip Size (0 + 1)",
              title_fontsize=12)

fig.subplots_adjust(left=0.01, right=0.95, top=0.97, bottom=0.01)
plt.show()
